# 05. Data Preprocessing

## Learning Objectives
- Understand missing value handling strategies
- Compare feature scaling methods
- Categorical variable encoding
- Handling imbalanced data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.datasets import load_iris, load_wine

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

## 1. Handling Missing Values

In [ ]:
# Create sample data with missing values
np.random.seed(42)
data = {
    'age': [25, 30, np.nan, 40, 35, np.nan, 50, 28],
    'income': [50000, np.nan, 60000, 80000, np.nan, 70000, 90000, 55000],
    'score': [85, 90, 75, np.nan, 88, 92, np.nan, 78]
}
df = pd.DataFrame(data)

print("Original data:")
print(df)
print(f"\nMissing value count:\n{df.isnull().sum()}")
print(f"\nMissing value ratio:\n{df.isnull().mean() * 100:.2f}%")

### 1.1 SimpleImputer - Basic Imputation Strategies

In [ ]:
# Impute with mean
imputer_mean = SimpleImputer(strategy='mean')
df_mean = pd.DataFrame(
    imputer_mean.fit_transform(df),
    columns=df.columns
)

# Impute with median
imputer_median = SimpleImputer(strategy='median')
df_median = pd.DataFrame(
    imputer_median.fit_transform(df),
    columns=df.columns
)

# Impute with most frequent value
imputer_frequent = SimpleImputer(strategy='most_frequent')
df_frequent = pd.DataFrame(
    imputer_frequent.fit_transform(df),
    columns=df.columns
)

# Impute with constant
imputer_constant = SimpleImputer(strategy='constant', fill_value=0)
df_constant = pd.DataFrame(
    imputer_constant.fit_transform(df),
    columns=df.columns
)

print("Mean imputation:")
print(df_mean)
print(f"\nMedian imputation (age column): {df_median['age'].values}")
print(f"Most frequent imputation (age column): {df_frequent['age'].values}")

### 1.2 KNNImputer - K-Nearest Neighbors Imputation

In [ ]:
# Why: KNN imputation uses feature correlations to estimate missing values,
# producing more realistic estimates than simple statistics. It is especially
# useful when features are correlated (e.g., age and income).
imputer_knn = KNNImputer(n_neighbors=3)
df_knn = pd.DataFrame(
    imputer_knn.fit_transform(df),
    columns=df.columns
)

print("KNN imputation:")
print(df_knn)

# Visualization comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (method, df_filled) in zip(axes, [
    ('Mean', df_mean), 
    ('Median', df_median), 
    ('KNN', df_knn)
]):
    ax.scatter(df_filled['age'], df_filled['income'], alpha=0.7, s=100)
    ax.set_xlabel('Age')
    ax.set_ylabel('Income')
    ax.set_title(f'{method} Imputation')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Feature Scaling

In [ ]:
# Create data with different scales
np.random.seed(42)
data_scale = {
    'age': np.random.randint(20, 60, 100),
    'income': np.random.randint(30000, 150000, 100),
    'score': np.random.uniform(0, 100, 100)
}
df_scale = pd.DataFrame(data_scale)

print("Original data statistics:")
print(df_scale.describe())

### 2.1 StandardScaler (Standardization)

In [ ]:
# StandardScaler: (x - mean) / std
scaler_standard = StandardScaler()
df_standard = pd.DataFrame(
    scaler_standard.fit_transform(df_scale),
    columns=df_scale.columns
)

print("StandardScaler result:")
print(df_standard.describe())
print(f"\nMean: {df_standard.mean().values}")
print(f"Standard deviation: {df_standard.std().values}")

### 2.2 MinMaxScaler (Normalization)

In [ ]:
# MinMaxScaler: (x - min) / (max - min)
scaler_minmax = MinMaxScaler(feature_range=(0, 1))
df_minmax = pd.DataFrame(
    scaler_minmax.fit_transform(df_scale),
    columns=df_scale.columns
)

print("MinMaxScaler result:")
print(df_minmax.describe())
print(f"\nMin: {df_minmax.min().values}")
print(f"Max: {df_minmax.max().values}")

### 2.3 RobustScaler (Robust to Outliers)

In [ ]:
# Why: RobustScaler uses median and IQR instead of mean and std, so outliers
# don't distort the scaling. Prefer it when your data has significant outliers
# that would pull StandardScaler's mean/std away from the bulk of the data.
scaler_robust = RobustScaler()
df_robust = pd.DataFrame(
    scaler_robust.fit_transform(df_scale),
    columns=df_scale.columns
)

print("RobustScaler result:")
print(df_robust.describe())

### 2.4 Scaler Comparison Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Add outlier
df_outlier = df_scale.copy()
df_outlier.loc[0, 'income'] = 500000  # Add outlier

scalers = [
    ('Original', df_outlier),
    ('StandardScaler', pd.DataFrame(StandardScaler().fit_transform(df_outlier), columns=df_outlier.columns)),
    ('MinMaxScaler', pd.DataFrame(MinMaxScaler().fit_transform(df_outlier), columns=df_outlier.columns)),
    ('RobustScaler', pd.DataFrame(RobustScaler().fit_transform(df_outlier), columns=df_outlier.columns))
]

for ax, (name, data) in zip(axes, scalers):
    ax.boxplot([data['age'], data['income'], data['score']], labels=['age', 'income', 'score'])
    ax.set_title(name)
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Categorical Encoding

In [ ]:
# Categorical data sample
data_cat = {
    'color': ['red', 'blue', 'green', 'red', 'blue', 'green', 'red'],
    'size': ['S', 'M', 'L', 'M', 'S', 'L', 'M'],
    'quality': ['good', 'excellent', 'poor', 'good', 'excellent', 'poor', 'good']
}
df_cat = pd.DataFrame(data_cat)

print("Categorical data:")
print(df_cat)

### 3.1 LabelEncoder

In [ ]:
# LabelEncoder: convert categories to integers
le_color = LabelEncoder()
df_cat['color_encoded'] = le_color.fit_transform(df_cat['color'])

print("LabelEncoder result:")
print(df_cat[['color', 'color_encoded']])
print(f"\nClasses: {le_color.classes_}")
print(f"Mapping: {dict(zip(le_color.classes_, le_color.transform(le_color.classes_)))}")

### 3.2 OneHotEncoder

In [ ]:
# Why: OneHotEncoding creates binary columns for each category, avoiding the
# false ordinal relationship that LabelEncoder introduces. Use it for nominal
# (unordered) features so the model doesn't assume blue < green < red.
ohe = OneHotEncoder(sparse_output=False)
color_onehot = ohe.fit_transform(df_cat[['color']])

# Convert to DataFrame
df_onehot = pd.DataFrame(
    color_onehot,
    columns=ohe.get_feature_names_out(['color'])
)

print("OneHotEncoder result:")
print(pd.concat([df_cat['color'], df_onehot], axis=1))

### 3.3 OrdinalEncoder

In [ ]:
# OrdinalEncoder: for ordinal categorical variables
oe = OrdinalEncoder(categories=[['poor', 'good', 'excellent']])
df_cat['quality_encoded'] = oe.fit_transform(df_cat[['quality']])

print("OrdinalEncoder result:")
print(df_cat[['quality', 'quality_encoded']])
print(f"\nOrder: poor(0) < good(1) < excellent(2)")

### 3.4 Pandas get_dummies

In [ ]:
# pandas get_dummies (convenient one-hot encoding)
df_dummies = pd.get_dummies(df_cat[['color', 'size']], prefix=['color', 'size'])

print("pd.get_dummies result:")
print(df_dummies.head())

# Prevent multicollinearity with drop_first=True
df_dummies_drop = pd.get_dummies(df_cat[['color', 'size']], prefix=['color', 'size'], drop_first=True)
print(f"\ndrop_first=True (shape: {df_dummies_drop.shape}):")
print(df_dummies_drop.head())

## 4. Feature Selection

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Load Iris data
iris = load_iris()
X, y = iris.data, iris.target

print(f"Original data: {X.shape}")
print(f"Feature names: {iris.feature_names}")

### 4.1 SelectKBest (Statistical Selection)

In [ ]:
# F-statistic based selection
selector_f = SelectKBest(score_func=f_classif, k=2)
X_kbest_f = selector_f.fit_transform(X, y)

# Mutual information based selection
selector_mi = SelectKBest(score_func=mutual_info_classif, k=2)
X_kbest_mi = selector_mi.fit_transform(X, y)

print("SelectKBest (F-statistic):")
scores_f = pd.DataFrame({
    'Feature': iris.feature_names,
    'Score': selector_f.scores_
}).sort_values('Score', ascending=False)
print(scores_f)

print("\nSelectKBest (Mutual Information):")
scores_mi = pd.DataFrame({
    'Feature': iris.feature_names,
    'Score': selector_mi.scores_
}).sort_values('Score', ascending=False)
print(scores_mi)

### 4.2 RFE (Recursive Feature Elimination)

In [ ]:
# RFE with Random Forest
estimator = RandomForestClassifier(n_estimators=50, random_state=42)
selector_rfe = RFE(estimator, n_features_to_select=2, step=1)
X_rfe = selector_rfe.fit_transform(X, y)

print("RFE result:")
rfe_result = pd.DataFrame({
    'Feature': iris.feature_names,
    'Selected': selector_rfe.support_,
    'Ranking': selector_rfe.ranking_
}).sort_values('Ranking')
print(rfe_result)

### 4.3 Feature Importance (Random Forest)

In [ ]:
# Random Forest feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importance = pd.DataFrame({
    'Feature': iris.feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'], importance['Importance'])
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance - Iris Dataset')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Handling Imbalanced Data

In [ ]:
from sklearn.datasets import make_classification

# Create imbalanced data (10:1 ratio)
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.9, 0.1],  # 90% vs 10%
    random_state=42
)

# Check class distribution
unique, counts = np.unique(y_imb, return_counts=True)
print("Class distribution:")
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} ({cnt/len(y_imb)*100:.1f}%)")

# Visualization
plt.figure(figsize=(8, 5))
plt.bar(['Class 0', 'Class 1'], counts, color=['skyblue', 'salmon'])
plt.ylabel('Count')
plt.title('Imbalanced Dataset Distribution')
plt.grid(True, alpha=0.3)
plt.show()

### 5.1 SMOTE Concept (Theory)

In [ ]:
# SMOTE (Synthetic Minority Over-sampling Technique) concept explanation
print("""
How SMOTE works:

1. For each sample in the minority class:
   - Find K nearest neighbors (typically k=5)
   
2. Linear interpolation with a randomly selected neighbor:
   - new_sample = sample + lambda * (neighbor - sample)
   - lambda is a random value between 0 and 1
   
3. Generate synthetic samples to augment the minority class

Advantages:
- Lower risk of overfitting (not simple duplication)
- Decision boundary becomes more generalized

Disadvantages:
- Can be sensitive to noise
- Limited effectiveness in high-dimensional data

Usage:
- pip install imbalanced-learn
- from imblearn.over_sampling import SMOTE
- smote = SMOTE(random_state=42)
- X_resampled, y_resampled = smote.fit_resample(X, y)
""")

### 5.2 Class Weight Adjustment

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42
)

# Without weights
clf_no_weight = LogisticRegression(random_state=42, max_iter=1000)
clf_no_weight.fit(X_train_imb, y_train_imb)
y_pred_no_weight = clf_no_weight.predict(X_test_imb)

# Why: 'balanced' automatically sets class weights inversely proportional to
# class frequency, so the minority class gets a higher penalty for misclassification.
# This forces the model to pay more attention to rare examples.
clf_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
clf_balanced.fit(X_train_imb, y_train_imb)
y_pred_balanced = clf_balanced.predict(X_test_imb)

print("=== Without Weights ===")
print(classification_report(y_test_imb, y_pred_no_weight))

print("\n=== With Balanced Weights ===")
print(classification_report(y_test_imb, y_pred_balanced))

## 6. Practical Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Create mixed data
data_mixed = {
    'age': [25, np.nan, 35, 40, 30, 45, np.nan, 28],
    'income': [50000, 60000, np.nan, 80000, 70000, 90000, 55000, np.nan],
    'city': ['Seoul', 'Busan', 'Seoul', 'Daegu', 'Busan', 'Seoul', 'Daegu', 'Busan'],
    'education': ['Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master'],
    'purchased': [0, 1, 1, 0, 1, 1, 0, 1]
}
df_mixed = pd.DataFrame(data_mixed)

X_mixed = df_mixed.drop('purchased', axis=1)
y_mixed = df_mixed['purchased']

print("Mixed data:")
print(df_mixed)

In [ ]:
# Why: ColumnTransformer applies different preprocessing to numeric vs categorical
# features in a single step. Combined with Pipeline, it ensures all transformations
# happen inside CV folds — no data leakage, and new data gets identical treatment.
numeric_features = ['age', 'income']
categorical_features = ['city', 'education']

# Numeric preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Train (using all data since it's small)
pipeline.fit(X_mixed, y_mixed)

# Predict on new data
new_data = pd.DataFrame({
    'age': [30],
    'income': [70000],
    'city': ['Seoul'],
    'education': ['Master']
})

prediction = pipeline.predict(new_data)
probability = pipeline.predict_proba(new_data)

print(f"\nPrediction: {prediction[0]}")
print(f"Probability: {probability[0]}")

## Summary

### Key Concepts

**Missing Value Handling:**
- **SimpleImputer**: Replace with mean, median, mode, or constant
- **KNNImputer**: K-nearest neighbor based imputation

**Feature Scaling:**
- **StandardScaler**: Mean 0, std 1 (assumes normal distribution)
- **MinMaxScaler**: Normalize to 0-1 range
- **RobustScaler**: Uses median and IQR (robust to outliers)

**Categorical Encoding:**
- **LabelEncoder**: Unordered classification (for target variables)
- **OneHotEncoder**: Convert to binary vectors (beware of multicollinearity)
- **OrdinalEncoder**: For ordinal categorical variables

**Imbalanced Data:**
- **SMOTE**: Generate synthetic samples (requires: imbalanced-learn)
- **class_weight**: Adjust model weights

### Next Steps
- Utilize Pipeline and ColumnTransformer
- Integrate cross-validation with preprocessing
- Apply to real-world projects